# Quy Trình Làm Sạch và Chuẩn Hóa Dữ Liệu

## Mục đích
Xử lý dữ liệu sản phẩm từ file `raw1.csv` để tạo ra dataset sạch, chuẩn hóa, sẵn sàng cho phân tích.

## Các bước xử lý (13 bước)

### 1. **Đọc dữ liệu**
   - Nạp dữ liệu từ `raw1.csv` với encoding UTF-8-sig
   - Kiểm tra số dòng, cột và danh sách các trường dữ liệu

### 2. **Xử lý cột SOLD** (đã bán)
   - Trích xuất số lượng từ chuỗi như "đã bán 883"
   - Giá trị mặc định: 0 nếu không có dữ liệu hoặc N/A

### 3. **Xử lý cột PRICE** (giá)
   - Loại bỏ dấu chấm phân cách hàng nghìn (1.234.567 → 1234567)
   - Chuyển thành số nguyên
   - Giá trị mặc định: 0 nếu không có dữ liệu

### 4. **Xử lý cột DISCOUNT_PERCENT** (chiết khấu)
   - Xóa ký tự '%' và '-'
   - Trích xuất giá trị phần trăm
   - Giá trị mặc định: 0 nếu không có dữ liệu

### 5. **Xử lý cột PRODUCT_NAME** (tên sản phẩm)
   - Chuyển thành chữ thường
   - Loại bỏ dấu tiếng Việt (mã hóa NFKD)
   - Xóa cụm "hàng chính hãng" ở cuối
   - Xóa ký tự đặc biệt, giữ lại chữ, số, dấu gạch ngang, ngoặc
   - Xóa khoảng trắng dư

### 6. **Xử lý cột FREESHIP** (miễn phí vận chuyển)
   - Chuẩn hóa giá trị: 1 (có miễn phí) hoặc 0 (không miễn phí)
   - Chấp nhận các giá trị: '1', 'true', 'yes', 'có'

### 7. **Xử lý cột CATEGORY** (danh mục)
   - Chuyển thành chữ thường
   - Loại bỏ dấu tiếng Việt (mã hóa NFKD)
   - Xóa ký tự đặc biệt
   - Giá trị mặc định: 'other' nếu rỗng

### 8. **Xử lý cột PRODUCT_RATING** (đánh giá sản phẩm)
   - Chuyển thành số thực (float)
   - Điền giá trị bị thiếu bằng trung vị (median)

### 9. **Xử lý cột NUMBER_OF_REVIEWS** (số lượng đánh giá)
   - Trích xuất số từ định dạng "(19)" → 19
   - Điền giá trị bị thiếu bằng trung vị (median)

### 10. **Xóa dòng không hợp lệ**
   - Xóa dòng có price = 0 (lỗi trong trích xuất)
   - Xóa dòng có sold < 0

### 11. **Xóa dòng trùng lặp**
   - Dựa trên kết hợp: (product_name, price, category)
   - Giữ lại dòng đầu tiên, xóa các dòng trùng sau

### 12. **Chuẩn hóa kiểu dữ liệu**
   - price, discount_percent, freeship, sold, number_of_reviews → int64
   - product_rating → float64
   - product_name, category → object (string)

### 13. **Lưu dữ liệu**
   - Xuất ra file `clean1.csv`
   - Sử dụng encoding UTF-8-sig (có BOM) để Excel hiển thị đúng font tiếng Việt

## Kết quả
- Dataset được làm sạch, chuẩn hóa
- Loại bỏ dữ liệu không hợp lệ và trùng lặp
- Sẵn sàng cho phân tích và visualization

In [3]:
import pandas as pd
import numpy as np
import re

print("=" * 70)
print("CHƯƠNG TRÌNH LÀM SẠCH VÀ CHUẨN HÓA DỮ LIỆU")
print("=" * 70)

print("\n[1/13] Đọc dữ liệu từ raw1.csv...")
df = pd.read_csv('raw1.csv', encoding='utf-8')
print(f"✓ Đã đọc {len(df)} dòng, {len(df.columns)} cột")

print("\n[2/13] Xử lý cột sold...")
def extract_sold(x):
    if pd.isna(x) or x == '' or x == 'N/A':
        return 0
    match = re.findall(r'\d+', str(x))
    if match:
        return int(match[-1])
    return 0

df['sold'] = df['sold'].apply(extract_sold)
print(f"✓ Xử lý xong")

print("\n[3/13] Xử lý cột price...")
def extract_price(x):
    if pd.isna(x) or x == '':
        return 0
    text = str(x).replace('.', '').strip()
    match = re.findall(r'\d+', text)
    if match:
        return int(''.join(match))
    return 0

df['price'] = df['price'].apply(extract_price)
print(f"✓ Xử lý xong")

print("\n[4/13] Xử lý cột discount_percent...")
def extract_discount(x):
    if pd.isna(x) or x == '' or x == 'N/A':
        return 0
    text = str(x).replace('%', '').replace('-', '').strip()
    match = re.findall(r'\d+', text)
    if match:
        return int(match[0])
    return 0

df['discount_percent'] = df['discount_percent'].apply(extract_discount)
print(f"✓ Xử lý xong")

print("\n[5/13] Xử lý cột product_name...")
def clean_product_name(x):
    if pd.isna(x):
        return ""
    text = str(x).lower()
    text = re.sub(r'\s*-\s*hang\s+chinh\s+hang\s*$', '', text)
    text = re.sub(r'[^\w\s\-\(\)]+', '', text)
    text = ' '.join(text.split())
    return text

df['product_name'] = df['product_name'].apply(clean_product_name)
print(f"✓ Xử lý xong")

print("\n[6/13] Xử lý cột freeship...")
def normalize_freeship(x):
    if pd.isna(x) or x == '' or x == 'N/A':
        return 0
    try:
        # Nếu là số (1.0, 1, 0, v.v.)
        val = float(x)
        return 1 if val > 0 else 0
    except (ValueError, TypeError):
        # Nếu là text
        val = str(x).lower().strip()
        if val in ['1', 'true', 'yes', 'co', 'có']:
            return 1
        else:
            return 0

df['freeship'] = df['freeship'].apply(normalize_freeship)
print(f"✓ Xử lý xong")

print("\n[7/13] Xử lý cột category...")
def clean_category(x):
    if pd.isna(x) or x == '':
        return 'other'
    text = str(x).lower()
    text = re.sub(r'[^\w\s]+', '', text)
    text = ' '.join(text.split())
    return text if text else 'other'

df['category'] = df['category'].apply(clean_category)
print(f"✓ Xử lý xong")

print("\n[8/13] Xử lý cột product_rating...")
df['product_rating'] = pd.to_numeric(df['product_rating'], errors='coerce')
median_rating = df['product_rating'].median()
df['product_rating'] = df['product_rating'].fillna(median_rating)
print(f"✓ Xử lý xong. Median rating: {median_rating:.2f}")

print("\n[9/13] Xử lý cột number_of_reviews...")
def extract_reviews(x):
    if pd.isna(x) or x == '' or x == 'N/A':
        return np.nan
    text = str(x).strip('()')
    match = re.findall(r'\d+', text)
    if match:
        return int(match[0])
    return np.nan

df['number_of_reviews'] = df['number_of_reviews'].apply(extract_reviews)
median_reviews = df['number_of_reviews'].median()
df['number_of_reviews'] = df['number_of_reviews'].fillna(median_reviews)
print(f"✓ Xử lý xong. Median reviews: {median_reviews:.0f}")

print("\n[10/13] Xóa dòng có giá trị thiếu ở price và sold...")
initial_rows = len(df)
df = df[df['price'] > 0]
df = df[df['sold'] >= 0]
removed_rows = initial_rows - len(df)
print(f"✓ Xóa {removed_rows} dòng. Còn lại {len(df)} dòng")

print("\n[11/13] Xóa dòng trùng lặp...")
initial_rows = len(df)
df = df.drop_duplicates(subset=['product_name', 'price', 'category'], keep='first')
removed_duplicates = initial_rows - len(df)
print(f"✓ Xóa {removed_duplicates} dòng trùng lặp. Còn lại {len(df)} dòng")

print("\n[12/13] Chuẩn hóa kiểu dữ liệu...")
df['price'] = df['price'].astype('int64')
df['discount_percent'] = df['discount_percent'].astype('int64')
df['product_rating'] = df['product_rating'].astype('float64')
df['number_of_reviews'] = df['number_of_reviews'].astype('int64')
df['freeship'] = df['freeship'].astype('int64')
df['sold'] = df['sold'].astype('int64')
df['product_name'] = df['product_name'].astype('object')
df['category'] = df['category'].astype('object')
print(f"✓ Chuẩn hóa xong")

print("\n[13/13] Lưu dữ liệu vào clean1.csv...")
df.to_csv('clean1.csv', index=False, encoding='utf-8-sig')
print(f"✓ Đã lưu {len(df)} dòng dữ liệu vào clean1.csv")
print(f"✓ File được lưu với encoding UTF-8-sig (có BOM) để Excel hiển thị đúng font tiếng Việt")

print("\n" + "=" * 70)
print("THỐNG KÊ KẾT QUẢ")
print("=" * 70)
print(f"\n📊 Số lượng dữ liệu: {len(df)} sản phẩm")
print(f"\n💰 Thống kê giá:")
print(f"   Min: {df['price'].min():,} VND | Max: {df['price'].max():,} VND | Mean: {df['price'].mean():.0f} VND")
print(f"\n📉 Thống kê chiết khấu:")
print(f"   Min: {df['discount_percent'].min()}% | Max: {df['discount_percent'].max()}% | Mean: {df['discount_percent'].mean():.1f}%")
print(f"\n⭐ Thống kê đánh giá:")
print(f"   Min: {df['product_rating'].min():.1f} | Max: {df['product_rating'].max():.1f} | Mean: {df['product_rating'].mean():.2f}")
print(f"\n📈 Thống kê lượng bán:")
print(f"   Min: {df['sold'].min()} | Max: {df['sold'].max()} | Mean: {df['sold'].mean():.0f}")
print(f"\n📂 Top 10 danh mục:")
for cat, count in df['category'].value_counts().head(10).items():
    print(f"   {cat}: {count}")

print("\n✅ Hoàn thành!")

CHƯƠNG TRÌNH LÀM SẠCH VÀ CHUẨN HÓA DỮ LIỆU

[1/13] Đọc dữ liệu từ raw1.csv...
✓ Đã đọc 1565 dòng, 8 cột

[2/13] Xử lý cột sold...
✓ Xử lý xong

[3/13] Xử lý cột price...
✓ Xử lý xong

[4/13] Xử lý cột discount_percent...
✓ Xử lý xong

[5/13] Xử lý cột product_name...
✓ Xử lý xong

[6/13] Xử lý cột freeship...
✓ Xử lý xong

[7/13] Xử lý cột category...
✓ Xử lý xong

[8/13] Xử lý cột product_rating...
✓ Xử lý xong. Median rating: 4.70

[9/13] Xử lý cột number_of_reviews...
✓ Xử lý xong. Median reviews: 3

[10/13] Xóa dòng có giá trị thiếu ở price và sold...
✓ Xóa 0 dòng. Còn lại 1565 dòng

[11/13] Xóa dòng trùng lặp...
✓ Xóa 293 dòng trùng lặp. Còn lại 1272 dòng

[12/13] Chuẩn hóa kiểu dữ liệu...
✓ Chuẩn hóa xong

[13/13] Lưu dữ liệu vào clean1.csv...
✓ Đã lưu 1272 dòng dữ liệu vào clean1.csv
✓ File được lưu với encoding UTF-8-sig (có BOM) để Excel hiển thị đúng font tiếng Việt

THỐNG KÊ KẾT QUẢ

📊 Số lượng dữ liệu: 1272 sản phẩm

💰 Thống kê giá:
   Min: 3,000 VND | Max: 28,600,000 VND |